# Pipeline Run Audit Logger

**Purpose:** Capture batch-level pipeline execution audit records for operational visibility and compliance.

**Target Table:** `workspace.audit.audit_pipeline_runs`

**Event Type:** Batch execution events

**Key Metrics:**
* Pipeline execution status (SUCCESS, FAILED, RUNNING)
* Data volume metrics (rows read/written)
* Runtime performance
* Error diagnostics

**Usage:**
```python
dbutils.notebook.run(
    "LMIP/notebooks/audit/audit_pipeline_runs",
    timeout_seconds=300,
    arguments={
        "run_id": "unique_run_identifier",
        "pipeline_name": "bronze_to_silver_customers",
        "environment": "PROD",
        "status": "SUCCESS",
        "rows_read": "10000",
        "rows_written": "9950",
        "runtime_seconds": "45.3"
    }
)
```

In [0]:
# Notebook parameters - configured via dbutils.widgets or job parameters
dbutils.widgets.text("run_id", "", "Run ID")
dbutils.widgets.text("pipeline_name", "", "Pipeline Name")
dbutils.widgets.text("environment", "DEV", "Environment (DEV/TEST/PROD)")
dbutils.widgets.text("status", "RUNNING", "Status (RUNNING/SUCCESS/FAILED)")
dbutils.widgets.text("rows_read", "0", "Rows Read")
dbutils.widgets.text("rows_written", "0", "Rows Written")
dbutils.widgets.text("runtime_seconds", "0", "Runtime (seconds)")
dbutils.widgets.text("error_message", "", "Error Message (if failed)")

# Retrieve parameter values
run_id = dbutils.widgets.get("run_id")
pipeline_name = dbutils.widgets.get("pipeline_name")
environment = dbutils.widgets.get("environment")
status = dbutils.widgets.get("status")
rows_read = int(dbutils.widgets.get("rows_read") or 0)
rows_written = int(dbutils.widgets.get("rows_written") or 0)
runtime_seconds = float(dbutils.widgets.get("runtime_seconds") or 0.0)
error_message = dbutils.widgets.get("error_message")

print(f"Audit Log Entry: {pipeline_name} | Run ID: {run_id} | Status: {status}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType, DecimalType
from datetime import datetime
import uuid

In [0]:
# Validation: Ensure mandatory parameters are provided
if not run_id:
    raise ValueError("Parameter 'run_id' is required")
    
if not pipeline_name:
    raise ValueError("Parameter 'pipeline_name' is required")
    
if status not in ["RUNNING", "SUCCESS", "FAILED"]:
    raise ValueError(f"Invalid status: {status}. Must be RUNNING, SUCCESS, or FAILED")
    
if environment not in ["DEV", "TEST", "PROD"]:
    raise ValueError(f"Invalid environment: {environment}. Must be DEV, TEST, or PROD")

print("✓ Parameter validation passed")

In [0]:
# Generate surrogate key
audit_run_sk = int(str(uuid.uuid4().int)[:18])  # 18-digit bigint

# Capture timestamps
current_timestamp = datetime.now()
start_time = current_timestamp if status == "RUNNING" else None
end_time = current_timestamp if status in ["SUCCESS", "FAILED"] else None

# Build audit record
audit_record = {
    "audit_run_sk": audit_run_sk,
    "run_id": run_id,
    "pipeline_name": pipeline_name,
    "environment": environment,
    "start_time": start_time or current_timestamp,
    "end_time": end_time,
    "status": status,
    "rows_read": rows_read,
    "rows_written": rows_written,
    "runtime_seconds": runtime_seconds,
    "error_message": error_message if error_message else None
}

print(f"Generated audit record with SK: {audit_run_sk}")

In [0]:
# Create DataFrame from audit record
audit_df = spark.createDataFrame([audit_record])

# Write to audit table (append mode)
target_table = "workspace.audit.audit_pipeline_runs"

try:
    audit_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(target_table)
    
    print(f"✓ Successfully logged audit record to {target_table}")
    print(f"  - Pipeline: {pipeline_name}")
    print(f"  - Run ID: {run_id}")
    print(f"  - Status: {status}")
    print(f"  - Rows Read: {rows_read:,}")
    print(f"  - Rows Written: {rows_written:,}")
    print(f"  - Runtime: {runtime_seconds:.2f}s")
    
except Exception as e:
    print(f"✗ Failed to write audit record: {str(e)}")
    raise

In [0]:
%sql
-- Verify the audit log was written successfully
-- Shows the 5 most recent pipeline run audit records

SELECT 
    run_id,
    pipeline_name,
    environment,
    status,
    start_time,
    end_time,
    rows_read,
    rows_written,
    runtime_seconds,
    SUBSTRING(error_message, 1, 50) as error_preview
FROM workspace.audit.audit_pipeline_runs
ORDER BY start_time DESC
LIMIT 5

In [0]:
# Return success indicator for orchestration
dbutils.notebook.exit({"status": "success", "audit_run_sk": audit_run_sk})